# Streamlit & Gradio — ML Demo UIs

In [ ]:
# Day 27 — Streamlit & Gradio
# In MLOps, building a great model or dashboard in Jupyter isn't enough.
# Stakeholders, SREs, and PMs need a web UI they can click and interact with —
# WITHOUT writing frontend HTML/CSS/React!
#
# Two Python-first tools solve this in minutes:
#   1. Streamlit -> Best for data dashboards, monitoring boards, internal tools
#   2. Gradio    -> Best for quick ML model / LLM demo interfaces
#
# Today we'll cover:
#   1. Streamlit basics (st.title, st.write, st.dataframe, st.metric)
#   2. Interactive widgets (st.slider, st.selectbox, st.file_uploader)
#   3. Plotting in Streamlit (st.pyplot, st.line_chart)
#   4. Gradio Interface & Blocks
#   5. Layout, sidebar, and deployment tips
#
# Install: pip install streamlit gradio
# NOTE: Both run as web servers from the terminal, so we'll write .py files here.

### 1. Streamlit Basics — Metrics & DataFrames

In [ ]:
# Streamlit turns a top-to-bottom Python script into a live web app.
# Every time a user interacts with a widget, Streamlit re-runs the script.

import os
os.makedirs('ui_demo', exist_ok=True)

st_basic = '''import streamlit as st
import pandas as pd
import numpy as np

# Page configuration (must be first Streamlit command)
st.set_page_config(page_title="Server Health Monitor", layout="wide")

st.title("Server Health & MLOps Dashboard")
st.write("Live overview of cluster metrics and resource utilization.")

# KPI Metric Cards in columns
col1, col2, col3 = st.columns(3)
col1.metric(label="Active Servers", value="24", delta="+2")
col2.metric(label="Avg CPU Load", value="64%", delta="-5%")
col3.metric(label="Error Rate", value="0.12%", delta="+0.02%", delta_color="inverse")

# Sample DataFrame
df = pd.DataFrame({
    "server": ["web-01", "web-02", "db-01", "ml-gpu-01"],
    "cpu": [45, 88, 72, 91],
    "memory": [60, 82, 85, 78],
    "status": ["healthy", "critical", "warning", "critical"]
})

st.subheader("Cluster Inventory")
st.dataframe(df, use_container_width=True)
'''

with open('ui_demo/st_basic.py', 'w') as f:
    f.write(st_basic)

print("Created ui_demo/st_basic.py")
print("Run in terminal: streamlit run ui_demo/st_basic.py")

### 2. Interactive Widgets & Plotting in Streamlit

In [ ]:
# Widgets let users filter data, upload CSVs, or adjust thresholds interactively.

st_interactive = '''import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

st.title("Interactive Log & Metric Explorer")

# Sidebar controls
st.sidebar.header("Filters")
cpu_threshold = st.sidebar.slider("CPU Alert Threshold (%)", min_value=50, max_value=100, value=80)
env = st.sidebar.selectbox("Environment", ["All", "prod", "staging", "dev"])

# Optional CSV upload
uploaded_file = st.sidebar.file_uploader("Upload Server Metrics CSV", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
else:
    df = pd.DataFrame({
        "server": ["web-01", "web-02", "api-01", "db-01", "worker-01"],
        "env": ["prod", "prod", "staging", "prod", "dev"],
        "cpu": [42, 89, 55, 93, 30]
    })

if env != "All":
    df = df[df["env"] == env]

alerts = df[df["cpu"] >= cpu_threshold]
st.warning(f"{len(alerts)} server(s) above {cpu_threshold}% CPU!")
st.dataframe(alerts)

# Matplotlib chart inside Streamlit
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#e74c3c" if c >= cpu_threshold else "#2ecc71" for c in df["cpu"]]
ax.bar(df["server"], df["cpu"], color=colors)
ax.axhline(cpu_threshold, color="red", linestyle="--")
ax.set_ylabel("CPU %")
ax.set_title("Server CPU vs Threshold")
st.pyplot(fig)
'''

with open('ui_demo/st_interactive.py', 'w') as f:
    f.write(st_interactive)

print("Created ui_demo/st_interactive.py")
print("Run: streamlit run ui_demo/st_interactive.py")

### 3. Gradio — Quick ML Model Demo UIs

In [ ]:
# Gradio is built specifically for ML/AI demos.
# You wrap ANY Python prediction function with gr.Interface(fn, inputs, outputs)
# and get a shareable web UI + REST API automatically!

gradio_demo = '''import gradio as gr

def predict_server_risk(cpu: float, memory: float, error_count: int) -> str:
    risk_score = cpu * 0.4 + memory * 0.3 + error_count * 1.2
    if risk_score > 75:
        return f"CRITICAL (Risk Score: {risk_score:.1f}) — Scale immediately!"
    elif risk_score > 45:
        return f"WARNING (Risk Score: {risk_score:.1f}) — Monitor closely."
    return f"HEALTHY (Risk Score: {risk_score:.1f}) — Operating normally."

# Simple Interface
demo = gr.Interface(
    fn=predict_server_risk,
    inputs=[
        gr.Slider(0, 100, value=50, label="CPU Usage (%)"),
        gr.Slider(0, 100, value=60, label="Memory Usage (%)"),
        gr.Number(value=5, label="Errors (last hour)")
    ],
    outputs=gr.Textbox(label="Health Prediction"),
    title="Server Risk Predictor",
    description="Adjust server metrics to test the risk scoring logic."
)

if __name__ == "__main__":
    demo.launch()
'''

with open('ui_demo/gradio_demo.py', 'w') as f:
    f.write(gradio_demo)

print("Created ui_demo/gradio_demo.py")
print("Run: python ui_demo/gradio_demo.py")
print()
print("Tip: gr.Blocks() gives you full layout control (Tabs, Rows, Columns)")
print("     just like Streamlit, while gr.Interface() is the 5-line quick demo.")

In [ ]:
# Practice Questions:
#
# 1. When would you choose Streamlit over Gradio (and vice versa)?
# 2. Why does st.set_page_config() have to be the first Streamlit call in a script?
# 3. How does @st.cache_data or @st.cache_resource help when loading large ML models?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Server Monitoring Dashboard (Streamlit)
# Build a Streamlit app with:
# - 3 st.metric() cards (Total Pods, Healthy %, Active Alerts)
# - A sidebar slider for CPU threshold and selectbox for namespace
# - A filtered DataFrame + Matplotlib bar chart showing pod CPU usage

# Exercise 2: ML Experiment Results Viewer
# Create a Streamlit app that reads an experiment CSV (model, accuracy, latency, cost)
# and lets the user filter by minimum accuracy and compare models side-by-side.

# Exercise 3: Deployment Status Board (Gradio Blocks)
# Using gr.Blocks() with gr.Row() and gr.Tab():
# - Tab 1: Predict server health from CPU, memory, disk, errors
# - Tab 2: Batch check a JSON/CSV list of servers and return a summary table